In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

from src.problems import (
    MMF1, MMF4, MMF11_L, MMF16_L3, MMF16_20,
    ZDT1, ZDT3, ZDT4, ZDT6,
    DTLZ1, DTLZ2, DTLZ3, DTLZ4, DTLZ7,
    WFG1, WFG2, WFG4, WFG5, WFG9,
    BBOB_Gallagher_Mock,
    evaluate_problem,
)
from src.visualization import display_pareto_fronts_catalog, display_decision_space

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Fitness Landscape – Catálogo Completo de Problemas

Cada classe de problema possui o método `true_pareto_front(n)` que retorna `(X, F)`.

A função `display_pareto_fronts_catalog(problem, F_landscape, ...)` mostra:
- **Cinza**: amostra do espaço de objetivos
- **Vermelho**: Pareto front teórico verdadeiro (do método da classe)
- **Azul**: Pareto front empírico (non-dominated sorting sobre os dados amostrados)

In [ ]:
VALID_METHODS = ['sobol', 'latin_hypercube', 'random']
METHOD_COLORS = {'sobol': 'red', 'latin_hypercube': 'dodgerblue', 'random': 'orange'}


def generate_samples(problem, n_samples=100_000, method='sobol'):
    """Sample the decision space and evaluate objectives.

    Uses a full grid for n_var <= 3 (regardless of method).
    For n_var > 3:
      'sobol'           - scrambled Sobol sequence; generates exactly 2^ceil(log2(n_samples)) points.
      'latin_hypercube' - Latin Hypercube Sampling (original behaviour).
      'random'          - uniform random sampling.
    """
    if method not in VALID_METHODS:
        raise ValueError(f"method must be one of {VALID_METHODS}, got '{method}'")
    n = problem.n_var
    if n <= 3:
        n_pts = max(2, int(round(n_samples ** (1.0 / n))))
        grids = [np.linspace(problem.xl[i], problem.xu[i], n_pts) for i in range(n)]
        mesh = np.meshgrid(*grids, indexing='ij')
        X = np.column_stack([m.ravel() for m in mesh])
    else:
        if method == 'sobol':
            m = int(np.ceil(np.log2(n_samples)))
            sampler = qmc.Sobol(d=n, scramble=True, seed=42)
            X = qmc.scale(sampler.random_base2(m), problem.xl, problem.xu)
        elif method == 'latin_hypercube':
            sampler = qmc.LatinHypercube(d=n, seed=42)
            X = qmc.scale(sampler.random(n_samples), problem.xl, problem.xu)
        elif method == 'random':
            rng = np.random.default_rng(42)
            X = rng.uniform(problem.xl, problem.xu, size=(n_samples, n))
    return X, evaluate_problem(problem, X)


def empirical_pareto_front(F):
    """Return the non-dominated front."""
    fronts = NonDominatedSorting().do(F)
    return F[fronts[0]]


DATA_DIR = 'data/dataframes'


def analyze_problem(problem, name, n_samples=100_000,
                    methods=None,
                    pair_vars=None,
                    true_color='white',
                    elev_offset=-15, azim_offset=225, roll_offset=0,
                    **plot_kwargs):
    """
    Pipeline completo de analise para um problema do catalogo.

    Para cada metodo de amostragem em `methods`:
        1. Amostragem do espaco de decisao (grid para n_var <= 3, metodo escolhido caso contrario)
        2. Salvamento em parquet (data/dataframes/{name}/df_{name}_landscape_{method}.parquet)
        3. Calculo do Pareto front empirico via non-dominated sorting
    Apos todos os metodos:
        4. Obtencao do Pareto front teorico via problem.true_pareto_front()
        5. Visualizacao conjunta do espaco de objetivos (display_pareto_fronts_catalog)
           com um PF empirico por metodo na legenda
        6. Visualizacao do espaco de decisoes (display_decision_space) com dados do primeiro metodo

    Parameters
    ----------
    problem : Problem
        Instancia de problema do catalogo (deve ter true_pareto_front()).
    name : str
        Nome do problema (usado como titulo e nome da pasta de saida).
    n_samples : int, default 100_000
        Numero de amostras a gerar por metodo. Para Sobol, o numero real
        e 2^ceil(log2(n_samples)).
    methods : list of str or None, default None
        Metodos de amostragem a utilizar. Valores validos: 'sobol',
        'latin_hypercube', 'random'. None = todos os tres.
    pair_vars : list of int or None, default None
        Variaveis de decisao (1-indexed) a incluir no grid pairwise.
        Ex: [1, 2, 20] -> grid 3x3 com x1, x2, x3.
        None = grid completo (todas as variaveis).
    true_color : str, default 'white'
        Cor dos pontos/linha do Pareto front/set teorico.
        'white' = preenchimento branco com contorno preto.
    elev_offset : float, default -15
        Offset de elevacao (eixo X) para graficos 3D.
        Elevacao final = 25 + elev_offset.
    azim_offset : float, default 225
        Offset de azimute (eixo Z) para graficos 3D.
        Azimute final = 45 + azim_offset.
    roll_offset : float, default 0
        Offset de roll (eixo Y) para graficos 3D.
    **plot_kwargs
        Parametros adicionais repassados a display_pareto_fronts_catalog
        (ex: xlim, ylim, zlim para limites manuais dos eixos).
    """
    if methods is None:
        methods = list(VALID_METHODS)

    x_cols = [f'x_{i+1}' for i in range(problem.n_var)]
    f_cols = [f'f{i+1}' for i in range(problem.n_obj)]
    out_dir = os.path.join(DATA_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    results = []
    X_first, F_first, pf_X_first = None, None, None

    for method in methods:
        print(f'\n[{method}] Gerando amostras para {name} '
              f'(n_var={problem.n_var}, n_obj={problem.n_obj})...')
        X, F = generate_samples(problem, n_samples, method=method)
        print(f'  Amostras geradas: {len(X):,}')

        df = pd.DataFrame(np.column_stack([X, F]), columns=x_cols + f_cols)
        path = os.path.join(out_dir, f'df_{name}_landscape_{method}.parquet')
        df.to_parquet(path)
        print(f'  Salvo em {path}')

        pf_idx = NonDominatedSorting().do(F)[0]
        pf_emp_F = F[pf_idx]
        pf_emp_X = X[pf_idx]
        print(f'  PF empirico: {len(pf_emp_F)} pontos')

        if X_first is None:
            X_first, F_first, pf_X_first = X, F, pf_emp_X

        results.append({'method': method, 'pf_F': pf_emp_F, 'pf_X': pf_emp_X})

    X_true, F_true = problem.true_pareto_front()
    print(f'\n  PF teorico: {len(F_true)} pontos')

    rot = dict(elev_offset=elev_offset, azim_offset=azim_offset, roll_offset=roll_offset)

    print('\n  -- Espaco de Objetivos --')
    display_pareto_fronts_catalog(
        problem, F_first,
        [r['pf_F'] for r in results],
        front_names=[f"PF {r['method']}" for r in results],
        front_colors=[METHOD_COLORS[r['method']] for r in results],
        sample_size=100_000,
        title=name,
        true_color=true_color,
        **rot, **plot_kwargs,
    )

    print('  -- Espaco de Decisoes --')
    first_color = METHOD_COLORS[results[0]['method']]
    display_decision_space(problem, X_first, F_first, X_true, pf_X_first, title=name,
                           pair_vars=pair_vars,
                           true_color=true_color, emp_color=first_color,
                           **rot)


---
## MMF – Multimodal Multi-objective Functions (CEC 2020)

### MMF1
Convex PF, 2 global PSs espelhados em x₁ = 2.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(MMF1(), 'MMF1')

### MMF4
Concave PF, 4 global PSs (niching simétrico).  
PF: f₂ = 1 − f₁²

In [ ]:
analyze_problem(MMF4(), 'MMF4')

### MMF11_L
1 global PS + 1 local PS.  
PF: f₂ = g*/f₁ (hipérbole)

In [ ]:
analyze_problem(MMF11_L(), 'MMF11_L')

### MMF16_L3
3-objetivo, 3 variáveis, 2 global PS + 2 local PS.  
PF: esfera de raio (1 + g*)

In [ ]:
analyze_problem(MMF16_L3(), 'MMF16_L3', 
                pair_vars=None, #Variáveis de decisão (1-indexed) a incluir no grid pairwise. Ex: [1, 2, 20] → grid 3x3 com x1, x2, x3. None = grid completo (todas as variáveis).
                true_color='red', #Cor dos pontos/linha do Pareto front/set teórico. 'white' = preenchimento branco com contorno preto.
                emp_color='white', #Cor dos pontos do Pareto front/set empírico.
                elev_offset=-27, #Offset de elevação (eixo X) para gráficos 3D. Elevação final = 25 + elev_offset.
                azim_offset=225, #Offset de azimute (eixo Z) para gráficos 3D. Azimute final = 45 + azim_offset.
                roll_offset=0) #Offset de roll (eixo Y) para gráficos 3D.
                #Parâmetros adicionais repassados a display_pareto_fronts_catalog (ex: xlim, ylim, zlim para limites manuais dos eixos).

### MMF16_20
3-objetivo, 20 variáveis (só x₂₀ participa de g).  
PF: esfera de raio (1 + g*)

In [ ]:
analyze_problem(MMF16_20(), 'MMF16_20', pair_vars=[1, 2, 20])

---
## ZDT – Zitzler, Deb, Thiele (2000)

### ZDT1
Convex PF, 30 variáveis.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(ZDT1(), 'ZDT1', 
                pair_vars=[1, 2], 
                true_color='white', 
                emp_color='red',
                elev_offset=-27, 
                azim_offset=225, 
                roll_offset=0)

### ZDT3
Disconnected PF (5 segmentos), 30 variáveis.  
PF: f₂ = 1 − √f₁ − f₁·sin(10πf₁)

In [ ]:
analyze_problem(ZDT3(), 'ZDT3', pair_vars=[1, 2])

### ZDT4
Multimodal g (muitos fronts locais), 10 variáveis.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(ZDT4(), 'ZDT4', ylim=(0, 5), pair_vars=[1, 2])

### ZDT6
Nonconvex, non-uniform density, 10 variáveis.  
PF: f₂ = 1 − f₁²

In [ ]:
analyze_problem(ZDT6(), 'ZDT6', pair_vars=[1, 2])

---
## DTLZ – Deb, Thiele, Laumanns, Zitzler (2001)

### DTLZ1
Linear PF, highly multimodal g, 7 variáveis.  
PF: f₁ + f₂ + f₃ = 0.5

In [ ]:
analyze_problem(DTLZ1(), 'DTLZ1', xlim=(0, 2), ylim=(0, 2), zlim=(0, 2), pair_vars=[1, 2, 3])

### DTLZ2
Spherical PF, 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ2(), 'DTLZ2', pair_vars=[1, 2, 3])

### DTLZ3
Spherical PF + DTLZ1’s multimodal g, 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ3(), 'DTLZ3', xlim=(0, 2), ylim=(0, 2), zlim=(0, 2), pair_vars=[1, 2, 3])

### DTLZ4
Biased density on spherical PF (α = 100), 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ4(), 'DTLZ4', pair_vars=[1, 2, 3])

### DTLZ7
Disconnected PF (4 patches), 22 variáveis.  
PF: f₃ = 6 − f₁(1+sin(3πf₁)) − f₂(1+sin(3πf₂))

In [ ]:
analyze_problem(DTLZ7(), 'DTLZ7', pair_vars=[1, 2, 3])

---
## WFG – Walking Fish Group (Huband et al. 2006)

### WFG1
Separable, unimodal, biased. PF: convex + mixed.

In [ ]:
analyze_problem(WFG1(), 'WFG1', pair_vars=[1, 2, 3])

### WFG2
Non-separable distance. PF: convex + disconnected.

In [ ]:
analyze_problem(WFG2(), 'WFG2', pair_vars=[1, 2, 3])

### WFG4
Multimodal (s_multi on every param). PF: concave.

In [ ]:
analyze_problem(WFG4(), 'WFG4', pair_vars=[1, 2, 3])

### WFG5
Deceptive (s_decept on every param). PF: concave.

In [ ]:
analyze_problem(WFG5(), 'WFG5', pair_vars=[1, 2, 3])

### WFG9
Parameter-dependent bias, deceptive + multimodal, nonseparable. PF: concave.

In [ ]:
analyze_problem(WFG9(), 'WFG9', pair_vars=[1, 2, 3])

---
## BBOB Gallagher Mock
101 picos gaussianos por objetivo, paisagem aleatória. Sem PF analítico.

In [ ]:
#analyze_problem(BBOB_Gallagher_Mock(), 'BBOB Gallagher Mock')